# Second runtime: trust tests in parallel

Runtime: **L4 GPU**. 1) Drive is mounted by the first cell (rclone with the RCLONE_DRIVE_TOKEN secret, else the Colab mount): it restores every model session 1 trained (MyDrive/thinkingcap-mvtf). 2) Run all. The worker runs `plan_second.txt` (fresh-question tests, retrain-seed checks); it only ADDS new results and new adapter names, never overwrites session 1's. When the queue is done, the last cell releases this runtime so it stops costing units.

In [ ]:
# Google Drive first: every model and score is restored from it and saved to it.
# Since 16 Nov Colab's own Drive sign-in fails for this account ("credential propagation was unsuccessful"),
# so when the RCLONE_DRIVE_TOKEN secret exists, Drive is mounted with rclone at the SAME path
# (/content/drive/MyDrive): nothing else changes. The token is read from Colab Secrets, passed only to the
# rclone process, never printed and never written to a file by this cell.
import os, subprocess, time
from pathlib import Path
token = None
try:
    from google.colab import userdata
    token = userdata.get('RCLONE_DRIVE_TOKEN')
except Exception as e:
    print('no RCLONE_DRIVE_TOKEN in Colab Secrets:', type(e).__name__)
if token:
    if subprocess.run('command -v rclone', shell=True, capture_output=True).returncode:
        subprocess.run('apt-get -qq install -y fuse3 >/dev/null 2>&1; cd /tmp && curl -sSLO https://downloads.rclone.org/v1.76.0/rclone-v1.76.0-linux-amd64.zip'
                       ' && unzip -qo rclone-v1.76.0-linux-amd64.zip && cp rclone-v1.76.0-linux-amd64/rclone /usr/local/bin/', shell=True, check=True)
    subprocess.run('fusermount3 -uz /content/drive/MyDrive 2>/dev/null; fusermount -uz /content/drive/MyDrive 2>/dev/null', shell=True)  # a re-run remounts cleanly
    Path('/content/drive/MyDrive').mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, RCLONE_CONFIG_GDRIVE_TYPE='drive', RCLONE_CONFIG_GDRIVE_SCOPE='drive', RCLONE_CONFIG_GDRIVE_TOKEN=token)
    RCLONE = subprocess.Popen(['rclone', 'mount', 'gdrive:', '/content/drive/MyDrive', '--vfs-cache-mode', 'writes',
                               '--dir-cache-time', '2h', '--poll-interval', '30s', '--vfs-read-chunk-size', '64M', '--drive-chunk-size', '64M',
                               '--rc', '--rc-no-auth', '--rc-addr', '127.0.0.1:5572',
                               '--log-file', '/content/rclone.log', '--log-level', 'NOTICE'], env=env,
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    del env, token
    for _ in range(60):
        if Path('/content/drive/MyDrive/thinkingcap-mvtf').exists():
            break
        time.sleep(1)
    ok = Path('/content/drive/MyDrive/thinkingcap-mvtf').exists()
    print('Drive mounted with rclone:', 'OK, thinkingcap-mvtf found' if ok else 'FAILED, see /content/rclone.log')
else:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
%env PYTHONUNBUFFERED=1
%env TRACK_SESSION=second
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d ThinkingCap-Test || git clone -q https://github.com/BottleCapAI/ThinkingCap-Test
%cd /content/ThinkingCap-Test
!test -f .installed || (pip -q install uv && uv pip install --system -q -r requirements.txt && touch .installed)

In [ ]:
!rm -rf /content/T && git clone -q --depth 1 -b mvt-fading https://github.com/arsenelemoigne/Test /content/T && cp /content/T/thinkingcap/*.py /content/T/thinkingcap/plan*.txt . && git -C /content/T log -1 --format='%h %s'

In [ ]:
import os
try:  # the OpenRouter key for the astra stage, from Colab's Secrets panel (never printed, never in a file)
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('OpenRouter key loaded from Colab Secrets (not shown)')
except Exception as e:
    print('no OpenRouter key from Colab Secrets:', type(e).__name__)
try:  # the Delos worker key (astra --as-provider delos), same rules
    from google.colab import userdata
    os.environ['DELOS_API_KEY'] = userdata.get('DELOS_API_KEY')
    print('Delos key loaded from Colab Secrets (not shown)')
except Exception as e:
    print('no Delos key from Colab Secrets:', type(e).__name__)
try:  # the user's Google AI Studio key (astra --as-provider google: Gemini 4 Argon), same rules
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('Google key loaded from Colab Secrets (not shown)')
except Exception as e:
    print('no Google key from Colab Secrets:', type(e).__name__)
!python track.py setup

In [ ]:
!python track.py plan --plan-file plan_line_a.txt --wait-hours 0.2 2>&1 | tee -a /content/track_log.txt

In [ ]:
# queue finished: flush Drive (wait for rclone's pending uploads, or Colab's own flush), then release this runtime
import json, time, subprocess, urllib.request
from google.colab import runtime
def rclone_pending():
    try:
        req = urllib.request.Request('http://127.0.0.1:5572/vfs/stats', data=b'{}', headers={'Content-Type': 'application/json'})
        d = json.load(urllib.request.urlopen(req, timeout=10)).get('diskCache', {})
        return d.get('uploadsInProgress', 0) + d.get('uploadsQueued', 0)
    except Exception:
        return None
pending = rclone_pending()
if pending is not None:
    t0 = time.time()
    while pending and time.time() - t0 < 7200:
        print(f'rclone: {pending} uploads pending', flush=True)
        time.sleep(30)
        pending = rclone_pending() or 0
    subprocess.run('fusermount3 -u /content/drive/MyDrive || fusermount -u /content/drive/MyDrive', shell=True)
    print('rclone: uploads done, Drive unmounted')
else:
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception as error:
        print(error)
time.sleep(60)
runtime.unassign()
